# 2pt N-State Fit: k0 1-state

Legacy-aligned notebook for the k0 SS one-state fits. The fit windows come directly from the legacy input files.

Edit the config block below, then run the notebook cells in order.


## Imports / Setup


In [1]:
from pathlib import Path
import sys

SRC_DIR = Path("/Users/xiang/Desktop/codes/lat-hadron-analysis/src")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_nstate_fit_input_text,
    run_nstate_fit_from_notebook,
    validate_nstate_notebook_config,
)


## User Inputs

The fields below mirror the current plain-text N-state fit input format.
The defaults point to realistic example data included in the repository.


In [2]:
workflow_config = {'title_pattern': 'l64c64a076_m140_fit_pz*',
 'ns': 64,
 'nt': 64,
 'lattice_spacing_fm': 0.076,

 # Data settings
 'c2pt': '/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/c2pt_csv/c2pt_5_5_k0_pz*_real.csv',
 'pzlist': [0, 1, 2],
 'fold_t': 'true',
 'model': 'symmetric',
 'fit_mode': 'correlated',

 # Fit settings
 'nstates': 1,
 'tmin_window': {0: [0, 23], 1: [0, 23], 2: [0, 13]},
 'tmax': {0: 25, 1: 25, 2: 15},
 'binsize': 10,
 'bootstrap_samples': 200,
 'seed': 2026,

 # Prior settings
 'pz0_ground_energy': 0.0539202,
 'fix_ground_energy_from_dispersion': False,
 'plot': True,
 'results_dir': '/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit/results_nstate_fit_1state'}
workflow_config


{'title_pattern': 'l64c64a076_m140_fit_pz*',
 'ns': 64,
 'nt': 64,
 'lattice_spacing_fm': 0.076,
 'c2pt': '/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/c2pt_csv/c2pt_5_5_k0_pz*_real.csv',
 'pzlist': [0, 1, 2],
 'fold_t': 'true',
 'model': 'symmetric',
 'fit_mode': 'correlated',
 'nstates': 1,
 'tmin_window': {0: [0, 23], 1: [0, 23], 2: [0, 13]},
 'tmax': {0: 25, 1: 25, 2: 15},
 'binsize': 10,
 'bootstrap_samples': 200,
 'seed': 2026,
 'pz0_ground_energy': 0.0539202,
 'fix_ground_energy_from_dispersion': False,
 'plot': True,
 'results_dir': '/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit/results_nstate_fit_1state'}

## Option Guide

Edit only `workflow_config` in the cell above for normal usage.
The keys are grouped by comments so data settings, fit settings, and output settings stay easy to scan.

### Data settings
- `title_pattern`: Output title pattern. Use `*` where the momentum index `pz` should be inserted.
- `ns`: Spatial lattice extent `Ns`.
- `nt`: Temporal lattice extent `Nt`.
- `lattice_spacing_fm`: Lattice spacing in fm for metadata and summaries.
- `c2pt`: Correlator CSV path or wildcard pattern. Keep `*` in the filename when using multiple `pz` values.
- `pzlist`: List of momentum indices to analyze, for example `[0]` or `[0, 1]`.
- `fold_t`: Time-folding mode before fitting.
  Choices: `"none"` or `False` = no folding; `"periodic"` or `True` = symmetric fold; `"antiperiodic"` = antisymmetric fold.
- `model`: Correlator model.
- `fit_mode`: Statistical error model used in the nonlinear fit.
  Choices: `"uncorrelated"` = diagonal fit with per-time-slice bootstrap standard deviations; `"correlated"` = full covariance fit using one shared covariance matrix built from the full bootstrap ensemble and reused for the mean fit and bootstrap fits. If the correlated fit fails for a sample or the window covariance cannot be factorized, the code falls back to a diagonal fit built from the covariance diagonal.

### Fit settings
- `nstates`: Number of states to fit. This template is locked to `1`.
- `tmin_window`: Preferred default notebook-facing fit-window form. Use a dictionary like `{0: [4, 12], 5: [6, 12]}` to set one `[tmin, tmax]` window per momentum. The notebook helper materializes this into the backend fit-window table format automatically.
- `binsize`: Integer configuration bin size. Use `1` for no binning.
- `bootstrap_samples`: Number of bootstrap resamples. `None` lets the backend choose automatically.
- `bootstrap_size`: Number of binned configurations drawn per bootstrap sample. `None` uses the backend default.
- `seed`: Random seed for reproducible bootstrap sampling.

### Prior settings
- `pz0_ground_energy`: Optional pz=0 ground-state energy in lattice units. When provided, it serves as the dispersion-reference input used by `fix_ground_energy_from_dispersion`.
- `fix_ground_energy_from_dispersion`: Optional boolean. When `true`, the nonlinear fit fixes the ground-state energy to that same dispersion target. This is the closest repository-native analogue to the legacy fixed-`E0` setup and is the recommended default when you trust the dispersion anchor.
- `plot`: Whether to generate plots automatically.
  Choices: `True` or `False`.
- `results_dir`: Output directory. If omitted or set to `None`, outputs go to the notebook working directory.

Practical note:
- When `tmin_window` is provided for a momentum, the fit scans `tmin` from 0 up to the window end while keeping that momentum's `tmax` fixed.
- The warm-start initial guess comes from the lower-state fit output for the same state count.
- The soft priors are disabled in this template because it is 1-state only.
- Fit tables include `fallback_uncorrelated_successes`, the number of bootstrap samples in a given `tmin` window that succeeded only after falling back from the correlated fit to a diagonal fit.


## Input Summary / Validation

This notebook follows the same single-config pattern as the TGEVP template.
Fields that belong to the plain-text input file are rendered below; notebook-only runtime fields such as `results_dir` stay in the same config for convenience.


In [3]:
print(pretty_print_config(workflow_config))
print(render_nstate_fit_input_text(workflow_config))
parsed_nstate = validate_nstate_notebook_config(workflow_config)
parsed_nstate


{
  "title_pattern": "l64c64a076_m140_fit_pz*",
  "ns": 64,
  "nt": 64,
  "lattice_spacing_fm": 0.076,
  "c2pt": "/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/c2pt_csv/c2pt_5_5_k0_pz*_real.csv",
  "pzlist": [
    0,
    1,
    2
  ],
  "fold_t": "true",
  "model": "symmetric",
  "fit_mode": "correlated",
  "nstates": 1,
  "tmin_window": {
    "0": [
      0,
      23
    ],
    "1": [
      0,
      23
    ],
    "2": [
      0,
      13
    ]
  },
  "tmax": {
    "0": 25,
    "1": 25,
    "2": 15
  },
  "binsize": 10,
  "bootstrap_samples": 200,
  "seed": 2026,
  "pz0_ground_energy": 0.0539202,
  "fix_ground_energy_from_dispersion": false,
  "plot": true,
  "results_dir": "/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit/results_nstate_fit_1state"
}
l64c64a076_m140_fit_pz* 64 64 0.076
c2pt /Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/c2pt_csv/c2pt_5_5_k0_pz*_real.csv
pzl

NStateFitInput(title_pattern='l64c64a076_m140_fit_pz*', ns=64, nt=64, lattice_spacing_fm=0.076, correlator_path_pattern='/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/c2pt_csv/c2pt_5_5_k0_pz*_real.csv', pzlist=(0, 1, 2), fold_t='periodic', tmax='/var/folders/lp/9lcqf1_j5mldhw2q7rg992w00000gn/T/lqcd_nstate_tmax_4c88rqpt/nstate_tmax.txt', model='symmetric', fit_mode='correlated', pz0_ground_energy=0.0539202, fix_ground_energy_from_dispersion=False, nstates=(1,), tmin_window='/var/folders/lp/9lcqf1_j5mldhw2q7rg992w00000gn/T/lqcd_nstate_tmin_windows_zdviwhny/nstate_tmin_window.txt', binsize=10, bootstrap_samples=200, bootstrap_size=None, seed=2026, low_state_prior_tmin=None, lambda_prior=1.0, make_plots=True, results_dir=PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit/results_nstate_fit_1state'))

## Run Analysis


In [4]:
nstate_outputs = run_nstate_fit_from_notebook(workflow_config)
for path in nstate_outputs:
    print(path)


/Users/xiang/Desktop/codes/lat-hadron-analysis/src/lqcd_analysis/two_point/effective_mass.py:165: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(samples, axis=0)
/Users/xiang/.local/python_vev/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


[nstate-fit] correlated window covariance setup used shrinkage_lambda=0.10 for tmin=0, tmax=25


[nstate-fit] correlated window covariance setup used shrinkage_lambda=0.10 for tmin=1, tmax=25


[nstate-fit] correlated window covariance setup used shrinkage_lambda=0.10 for tmin=2, tmax=25


[nstate-fit] correlated window covariance setup used shrinkage_lambda=0.10 for tmin=3, tmax=25


...

[nstate-fit] correlated window covariance setup used shrinkage_lambda=0.10 for tmin=10, tmax=15


[nstate-fit] correlated window covariance setup used shrinkage_lambda=0.10 for tmin=11, tmax=15


[nstate-fit] correlated window covariance setup used shrinkage_lambda=0.10 for tmin=12, tmax=15


[nstate-fit] correlated window covariance setup used shrinkage_lambda=0.10 for tmin=13, tmax=15


/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit/results_nstate_fit_1state/l64c64a076_m140_fit_pz0/tables/l64c64a076_m140_fit_pz0_symmetric_correlator_mean.txt
/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit/results_nstate_fit_1state/l64c64a076_m140_fit_pz0/tables/l64c64a076_m140_fit_pz0_symmetric_effective_mass_tmax25.txt
/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit/results_nstate_fit_1state/l64c64a076_m140_fit_pz0/l64c64a076_m140_fit_pz0_symmetric_summary.txt
/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit/results_nstate_fit_1state/l64c64a076_m140_fit_pz0/plots/l64c64a076_m140_fit_pz0_symmetric_effective_mass_tmax25.pdf
/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit/results_nstate_fit_1state/l64c64a076_m140_fit_pz0/plots/l64c64a076_m140_fit_pz0_symme

## Inspect Outputs

The fit writes tables, bootstrap samples, plots, and a plotting notebook under `examples/outputs/`.


In [5]:
for path in nstate_outputs:
    print(Path(path).name)


l64c64a076_m140_fit_pz0_symmetric_correlator_mean.txt
l64c64a076_m140_fit_pz0_symmetric_effective_mass_tmax25.txt
l64c64a076_m140_fit_pz0_symmetric_summary.txt
l64c64a076_m140_fit_pz0_symmetric_effective_mass_tmax25.pdf
l64c64a076_m140_fit_pz0_symmetric_1state_energies_tmax25.pdf
l64c64a076_m140_fit_pz0_symmetric_1state_amplitudes_tmax25.pdf
l64c64a076_m140_fit_pz0_symmetric_1state_reconstruction_tmax25.pdf
l64c64a076_m140_fit_pz0_symmetric_nstate_plots.ipynb
l64c64a076_m140_fit_pz1_symmetric_correlator_mean.txt
l64c64a076_m140_fit_pz1_symmetric_effective_mass_tmax25.txt
l64c64a076_m140_fit_pz1_symmetric_summary.txt
l64c64a076_m140_fit_pz1_symmetric_effective_mass_tmax25.pdf
l64c64a076_m140_fit_pz1_symmetric_1state_energies_tmax25.pdf
l64c64a076_m140_fit_pz1_symmetric_1state_amplitudes_tmax25.pdf
l64c64a076_m140_fit_pz1_symmetric_1state_reconstruction_tmax25.pdf
l64c64a076_m140_fit_pz1_symmetric_nstate_plots.ipynb
l64c64a076_m140_fit_pz2_symmetric_correlator_mean.txt
l64c64a076_m140_fi